In [1]:
import pandas as pd
import numpy as np

np.random.seed(42)

n = 800
severities = ['LOW', 'MEDIUM', 'HIGH', 'CRITICAL']

data = pd.DataFrame({
    'severity': np.random.choice(severities, n, p=[0.3, 0.35, 0.25, 0.10]),
    'needs_icu': np.random.choice([1, 0], n, p=[0.3, 0.7]),
    'needs_trauma': np.random.choice([1, 0], n, p=[0.2, 0.8]),
    'hospital_has_icu': np.random.choice([1, 0], n, p=[0.5, 0.5]),
    'hospital_has_trauma': np.random.choice([1, 0], n, p=[0.4, 0.6]),
    'available_beds': np.random.randint(0, 30, n),
    'distance_km': np.round(np.random.uniform(0.5, 25, n), 2),
})

icu_ok = (data['needs_icu'] == 0) | (data['hospital_has_icu'] == 1)
trauma_ok = (data['needs_trauma'] == 0) | (data['hospital_has_trauma'] == 1)
has_capacity = data['available_beds'] > 0
capability_match = (icu_ok & trauma_ok & has_capacity).astype(float)

distance_penalty = data['distance_km'] / 25
bed_bonus = np.clip(data['available_beds'] / 30, 0, 1) * 0.2

data['match_score'] = np.where(
    capability_match == 1,
    np.clip(1.0 - distance_penalty + bed_bonus + np.random.normal(0, 0.03, n), 0, 1.2),
    0.0
)

data.head(10)

,severity,needs_icu,needs_trauma,hospital_has_icu,hospital_has_trauma,available_beds,distance_km,match_score
0,MEDIUM,0,1,1,0,9,10.13,0.000000
1,CRITICAL,1,0,1,0,20,6.30,0.862271
2,HIGH,0,0,1,1,4,2.88,0.961997
3,MEDIUM,0,0,1,0,26,4.89,1.007734
4,LOW,0,1,0,0,22,20.04,0.000000
5,LOW,0,0,0,0,19,17.10,0.422240
6,LOW,0,1,1,0,28,13.90,0.000000
7,HIGH,0,0,1,0,19,12.14,0.562447
8,MEDIUM,0,0,0,0,24,23.10,0.227546
9,HIGH,1,0,1,0,5,2.28,0.945088


In [2]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

X = data[['severity', 'needs_icu', 'needs_trauma', 'hospital_has_icu', 'hospital_has_trauma', 'available_beds', 'distance_km']]
y = data['match_score']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

preprocessor = ColumnTransformer(transformers=[
    ('cat', OneHotEncoder(handle_unknown='ignore'), ['severity']),
], remainder='passthrough')

model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(n_estimators=100, random_state=42)),
])

model.fit(X_train, y_train)

predictions = model.predict(X_test)
mae = mean_absolute_error(y_test, predictions)
print(f"Mean Absolute Error: {mae:.4f}")

Mean Absolute Error: 0.0407


In [3]:
import joblib

joblib.dump(model, '../trained_models/hospital_recommendation_model.pkl')
print("Model saved successfully!")

# Test: CRITICAL case needing ICU, hospital has ICU, plenty of beds, close by
test_case = pd.DataFrame({
    'severity': ['CRITICAL'],
    'needs_icu': [1],
    'needs_trauma': [0],
    'hospital_has_icu': [1],
    'hospital_has_trauma': [0],
    'available_beds': [15],
    'distance_km': [3.0],
})
print(f"Good match (has ICU, close, beds available): {model.predict(test_case)[0]:.3f}")

# Compare: same emergency, but hospital has NO ICU and NO beds
test_case_2 = pd.DataFrame({
    'severity': ['CRITICAL'],
    'needs_icu': [1],
    'needs_trauma': [0],
    'hospital_has_icu': [0],
    'hospital_has_trauma': [0],
    'available_beds': [0],
    'distance_km': [3.0],
})
print(f"Bad match (no ICU, no beds): {model.predict(test_case_2)[0]:.3f}")

Model saved successfully!
Good match (has ICU, close, beds available): 0.980
Bad match (no ICU, no beds): 0.047
